In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # Initialize BM25
        tokenized_corpus = [doc.lower().split(" ") for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # Initialize SBERT
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.sbert_embeddings = self.sbert_model.encode(corpus, convert_to_tensor=True)

    def _get_bm25_scores(self, query: str) -> dict:
        tokenized_query = query.lower().split(" ")
        doc_scores = self.bm25.get_scores(tokenized_query)
        return {i: score for i, score in enumerate(doc_scores)}

    def _get_sbert_scores(self, query: str) -> dict:
        query_embedding = self.sbert_model.encode(query, convert_to_tensor=True)
        cosine_scores = cosine_similarity(query_embedding.cpu().numpy().reshape(1, -1), self.sbert_embeddings.cpu().numpy())[0]
        return {i: score for i, score in enumerate(cosine_scores)}

    def _reciprocal_rank_fusion(self, bm25_ranks: dict, sbert_ranks: dict, k: int = 60, c: int = 60) -> dict:
        fused_scores = {}

        for doc_id, rank in bm25_ranks.items():
            fused_scores[doc_id] = fused_scores.get(doc_id, 0) + (1 / (c + rank))

        for doc_id, rank in sbert_ranks.items():
            fused_scores[doc_id] = fused_scores.get(doc_id, 0) + (1 / (c + rank))

        return fused_scores

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # Get BM25 scores and convert to ranks
        bm25_scores = self._get_bm25_scores(query)
        bm25_sorted_docs = sorted(bm25_scores.items(), key=lambda item: item[1], reverse=True)
        bm25_ranks = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(bm25_sorted_docs)}

        # Get SBERT scores and convert to ranks
        sbert_scores = self._get_sbert_scores(query)
        sbert_sorted_docs = sorted(sbert_scores.items(), key=lambda item: item[1], reverse=True)
        sbert_ranks = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(sbert_sorted_docs)}

        # Apply RRF
        fused_scores = self._reciprocal_rank_fusion(bm25_ranks, sbert_ranks, k=self.k)

        # Sort by fused score and get top_k documents
        sorted_fused_docs = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)

        results = []
        for rank, (doc_id, rrf_score) in enumerate(sorted_fused_docs[:top_k]):
            results.append({
                "doc_id": doc_id,
                "rrf_score": rrf_score,
                "bm25_rank": bm25_ranks.get(doc_id, len(self.corpus) + 1), # Assign a very low rank if not found
                "sbert_rank": sbert_ranks.get(doc_id, len(self.corpus) + 1), # Assign a very low rank if not found
                "text": self.corpus[doc_id]
            })
        return results

# Example usage:
retriever = HybridRetriever(corpus)
query = "What is self-attention?"
results = retriever.retrieve(query, top_k=3)

print(f"\nHybrid Retrieval Results for query: '{query}'")
for i, result in enumerate(results):
    print(f"Top {i+1} Document (ID: {result['doc_id']}, RRF Score: {result['rrf_score']:.4f}, BM25 Rank: {result['bm25_rank']}, SBERT Rank: {result['sbert_rank']}): {result['text']}")

In [ ]:
!pip install -U rank_bm25 sentence-transformers

In [ ]:
# Part 1: Document Corpus Setup

corpus = [
    "Transformers are a type of neural network architecture primarily used in natural language processing (NLP) tasks. They rely on self-attention mechanisms to weigh the importance of different words in a sequence.",
    "The attention mechanism in neural networks allows the model to focus on specific parts of the input sequence when making predictions. This is crucial for handling long-range dependencies efficiently.",
    "Self-attention computes a weighted sum of all other positions in the sequence, where the weight is determined by a learned compatibility function. This enables the model to capture contextual relationships.",
    "Recurrent Neural Networks (RNNs) process sequences by maintaining a hidden state that is updated at each step, making them suitable for time-series data and language modeling, though they struggle with long dependencies.",
    "Convolutional Neural Networks (CNNs) are widely used in computer vision for tasks like image classification and object detection, leveraging convolutional layers to extract hierarchical features.",
    "Gradient descent is an optimization algorithm used to minimize the loss function of a model by iteratively adjusting parameters in the direction opposite to the gradient of the loss function.",
    "Stochastic Gradient Descent (SGD) is a variant of gradient descent that computes the gradient and updates parameters using a small random subset of the training data, leading to faster but noisier updates.",
    "Adam (Adaptive Moment Estimation) is an optimization algorithm that combines the advantages of AdaGrad and RMSProp, providing adaptive learning rates for each parameter based on their first and second moments.",
    "Reinforcement learning involves an agent learning to make decisions by performing actions in an environment to maximize a cumulative reward. Key components include states, actions, rewards, and policies.",
    "Generative Adversarial Networks (GANs) consist of two neural networks, a generator and a discriminator, competing against each other to generate realistic data, commonly used for image synthesis.",
    "Transfer learning is a machine learning technique where a model trained on one task is re-purposed for a second related task, often by fine-tuning the pre-trained model on a new dataset."
]

print(f"Corpus created with {len(corpus)} documents.")
for i, doc in enumerate(corpus):
    print(f"Document {i+1}: {doc}")

In [ ]:
with open('/content/Assignment_AdvancedRAG.md', 'r') as f:
    assignment_content = f.read()
print(assignment_content)